In [1]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from notebookutils import *
from pyspark.sql import functions as F
from delta.tables import *
from notebookutils import mssparkutils

StatementMeta(, 62e4a13b-dfdc-49bb-8d4d-0e665331763a, 3, Finished, Available, Finished, False)

In [2]:
tables_str = tables_str
bronze = bronze
lakehouse = lakehouse
workspace = workspace
silver = silver

Tables = [t.strip() for t in tables_str.split(",")]

StatementMeta(, 62e4a13b-dfdc-49bb-8d4d-0e665331763a, 4, Finished, Available, Finished, False)

In [3]:
table = None
for i in Tables:
    if i.lower() == "observation":
        table = i
        break

if table is None:
    mssparkutils.notebook.exit("Exiting notebook: Table is not observation")

print(table + " table is present ")

StatementMeta(, 62e4a13b-dfdc-49bb-8d4d-0e665331763a, 5, Finished, Available, Finished, False)

Observation table is present 


In [4]:
bronze_path = f"Files/{bronze}/{table.lower()}"
silver_path = f"Files/{silver}/{table.lower()}"

# Check if silver path exists
silver_exists = mssparkutils.fs.exists(silver_path)

# Read bronze data
df = spark.read.format("delta").load(bronze_path)

# Apply filter only if silver exists
if silver_exists:
    df = df.filter(
        col("load_date") >= date_sub(current_date(), 4)
    )

df.printSchema()

StatementMeta(, 62e4a13b-dfdc-49bb-8d4d-0e665331763a, 6, Finished, Available, Finished, False)

root
 |-- api_url: string (nullable = true)
 |-- extraction_timestamp: string (nullable = true)
 |-- load_date: string (nullable = true)
 |-- raw_json: struct (nullable = true)
 |    |-- entry: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- link: string (nullable = true)
 |    |-- meta: string (nullable = true)
 |    |-- resourceType: string (nullable = true)
 |    |-- total: string (nullable = true)
 |    |-- type: string (nullable = true)



In [66]:
# 2. Clean outer brackets
df1 = df.withColumn(
    "entry_str",
    regexp_replace(col("raw_json.entry"), r"^\[|\]$", "")
)

# 3. Split into individual records
df2 = df1.withColumn(
    "entry_array",
    split(col("entry_str"), r"\}, \{")
)

# 4. Explode into rows
df3 = df2.withColumn(
    "entry_item",
    explode(col("entry_array"))
)

# 5. Extract fields with precise regex
df_final = df3.select(
    # IDs
    regexp_extract(col("entry_item"), r"id=([0-9]+)", 1).alias("observation_id"),
    regexp_extract(col("entry_item"), r"Patient/([0-9]+)", 1).alias("patient_id"),

    # Clinical info
    regexp_extract(col("entry_item"), r"display=([^,}]+)", 1).alias("test_name"),
    regexp_extract(col("entry_item"), r"coding=\[\{code=([0-9\\-]+)", 1).alias("loinc_code"),
    regexp_extract(col("entry_item"), r"coding=\[\{code=[^,}]+, system=([^,}]+)", 1).alias("coding_system"),

    # Measurement
    regexp_extract(col("entry_item"), r"value=([0-9]+)", 1).cast("double").alias("value"),
    regexp_extract(col("entry_item"), r"unit=([^,}]+)", 1).alias("unit"),

    # Status
    regexp_extract(col("entry_item"), r"status=([^,}]+)", 1).alias("status"),
    regexp_extract(col("entry_item"), r"resourceType=([^,}]+)", 1).alias("resourceType"),

    # Meta block (STRICT extraction)
    regexp_extract(col("entry_item"), r"lastUpdated=([^,}]+)", 1).alias("last_updated"),
    regexp_extract(col("entry_item"), r"versionId=([^,}]+)", 1).alias("version_id"),
    regexp_extract(col("entry_item"), r"meta=\{[^}]*source=([^,}]+)", 1).alias("source")
)

# 6. Show result
display(df_final.limit(100))
silver_condition = df_final

StatementMeta(, b9c006c8-0af5-4eb0-a0f1-2bfaf929f1f2, 68, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ff0ed87-38f2-44a7-9cab-4bc921cefa9a)

In [67]:

# -----------------------------
# Step 1: Ensure timestamp type
# -----------------------------
silver_encounter = silver_condition.withColumn(
    "last_updated",
    to_timestamp(col("last_updated"))
)

# -----------------------------
# Step 2: Define window (latest record per encounter + patient)
# -----------------------------
window_spec = Window.partitionBy(
    "observation_id"
).orderBy(
    col("last_updated").desc()
)

# -----------------------------
# Step 3: Deduplicate using row_number
# -----------------------------
silver_encounter = (
    silver_encounter
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

# -----------------------------
# Step 4: Display result (Fabric-safe)
# -----------------------------
display(silver_encounter.limit(5))

StatementMeta(, b9c006c8-0af5-4eb0-a0f1-2bfaf929f1f2, 69, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, acf8ceea-dce6-44ec-a592-2e86bd535a53)

In [68]:
staging_df = silver_encounter \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

StatementMeta(, b9c006c8-0af5-4eb0-a0f1-2bfaf929f1f2, 70, Finished, Available, Finished, False)

In [69]:
silver_path = f"Files/{silver}/{table.lower()}"

if not mssparkutils.fs.exists(silver_path):
    staging_df.write.format("delta").mode("overwrite").save(silver_path)

    mssparkutils.notebook.exit("Exiting notebook: First time load completed")
else:
    print("Table exists")

StatementMeta(, b9c006c8-0af5-4eb0-a0f1-2bfaf929f1f2, 71, Finished, Available, Finished, False)

In [70]:
staging_df.createOrReplaceTempView("staging_observation")

StatementMeta(, b9c006c8-0af5-4eb0-a0f1-2bfaf929f1f2, 72, Finished, Available, Finished, False)

In [71]:
query = f"""
MERGE INTO delta.`{silver_path}` AS tgt
USING staging_observation AS src
ON tgt.observation_id = src.observation_id
AND tgt.is_current = true

-- 1. EXPIRE OLD RECORD WHEN DATA CHANGES
WHEN MATCHED AND (
    COALESCE(tgt.patient_id, '') <> COALESCE(src.patient_id, '') OR
    COALESCE(tgt.test_name, '') <> COALESCE(src.test_name, '') OR
    COALESCE(tgt.loinc_code, '') <> COALESCE(src.loinc_code, '') OR
    COALESCE(tgt.coding_system, '') <> COALESCE(src.coding_system, '') OR
    COALESCE(tgt.value, -1) <> COALESCE(src.value, -1) OR
    COALESCE(tgt.unit, '') <> COALESCE(src.unit, '') OR
    COALESCE(tgt.status, '') <> COALESCE(src.status, '') OR
    COALESCE(tgt.resourceType, '') <> COALESCE(src.resourceType, '') OR
    COALESCE(tgt.last_updated, '') <> COALESCE(src.last_updated, '') OR
    COALESCE(tgt.version_id, '') <> COALESCE(src.version_id, '') OR
    COALESCE(tgt.source, '') <> COALESCE(src.source, '')
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()

-- 2. INSERT NEW RECORD IF NOT MATCHED
WHEN NOT MATCHED
THEN INSERT (
    observation_id,
    patient_id,
    test_name,
    loinc_code,
    coding_system,
    value,
    unit,
    status,
    resourceType,
    last_updated,
    version_id,
    source,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    src.observation_id,
    src.patient_id,
    src.test_name,
    src.loinc_code,
    src.coding_system,
    src.value,
    src.unit,
    src.status,
    src.resourceType,
    src.last_updated,
    src.version_id,
    src.source,
    current_timestamp(),
    NULL,
    true
)
"""

query1 = f"""
INSERT INTO delta.`{silver_path}` (
    observation_id,
    patient_id,
    test_name,
    loinc_code,
    coding_system,
    value,
    unit,
    status,
    resourceType,
    last_updated,
    version_id,
    source,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT 
    src.observation_id,
    src.patient_id,
    src.test_name,
    src.loinc_code,
    src.coding_system,
    src.value,
    src.unit,
    src.status,
    src.resourceType,
    src.last_updated,
    src.version_id,
    src.source,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_observation src
JOIN delta.`{silver_path}` tgt
    ON tgt.observation_id = src.observation_id
WHERE tgt.is_current = false
AND (
    COALESCE(tgt.patient_id, '') <> COALESCE(src.patient_id, '') OR
    COALESCE(tgt.test_name, '') <> COALESCE(src.test_name, '') OR
    COALESCE(tgt.loinc_code, '') <> COALESCE(src.loinc_code, '') OR
    COALESCE(tgt.coding_system, '') <> COALESCE(src.coding_system, '') OR
    COALESCE(tgt.value, -1) <> COALESCE(src.value, -1) OR
    COALESCE(tgt.unit, '') <> COALESCE(src.unit, '') OR
    COALESCE(tgt.status, '') <> COALESCE(src.status, '') OR
    COALESCE(tgt.resourceType, '') <> COALESCE(src.resourceType, '') OR
    COALESCE(tgt.last_updated, '') <> COALESCE(src.last_updated, '') OR
    COALESCE(tgt.version_id, '') <> COALESCE(src.version_id, '') OR
    COALESCE(tgt.source, '') <> COALESCE(src.source, '')
)
AND NOT EXISTS (
    SELECT 1
    FROM delta.`{silver_path}` t2
    WHERE t2.observation_id = src.observation_id
      AND t2.is_current = true
)
"""

spark.sql(query)
spark.sql(query1)

StatementMeta(, b9c006c8-0af5-4eb0-a0f1-2bfaf929f1f2, 73, Finished, Available, Finished, False)

DataFrame[]

In [ ]:
df = spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`") \
    .orderBy("version", ascending=False) \
    .limit(2)

display(df)

StatementMeta(, , -1, SessionError, , SessionError, True)

InvalidHttpRequest: [TooManyRequestsForCapacity] This spark job can't be run because you have hit a spark compute or API rate limit. To run this spark job, cancel an active Spark job through the Monitoring hub, choose a larger capacity SKU, or try again later. HTTP status code: 430 {Learn more} HTTP status code: 430.